# 02_2stage — detection backbone comparison on Colab

Trains the 10-backbone YOLOv8 detection comparison across all 6 subsets: DIVA-HisDB
(CB55, CS18, CS863) and U-DIADS-TL (Latin14396, Latin2, Syr341) — **60 runs**.

Layout: code from GitHub, data and checkpoints from Drive, hot paths on the local Colab
SSD. `00_data`, `80_models` and `99_evaluation` are recreated as symlinks at the repo
root, so every relative path in the configs resolves unchanged.

Expected on Drive:

    MyDrive/Thesis/
      00_data/                            <- same structure as the repo's 00_data
      80_models/02_2stage/pretrained/     <- optional yolov8n.pt
      80_models/                          <- receives weights/best.pt + results.csv

Safe to re-run: finished runs are pulled back from Drive and skipped.

## 1. Config

In [ ]:
# ----- edit these if your Drive layout differs -----------------------------
REPO_URL    = "https://github.com/Maserezio/fsl-tl-manuscripts.git"
REPO_BRANCH = "main"
DRIVE_ROOT  = "/content/drive/MyDrive/Thesis"   # holds 00_data/ and 80_models/
REPO_DIR    = "/content/repo"                   # code; re-cloned every session
WORK_DIR    = "/content/work"                   # local SSD: 00_data / 80_models / 99_evaluation
AUTOSYNC_MINUTES = 15                           # background push cadence; 0 disables

# --------------------------------------------------------------------------

import os, shutil, subprocess, sys, time, threading
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))

DRIVE_ROOT, REPO_DIR, WORK_DIR = map(Path, (DRIVE_ROOT, REPO_DIR, WORK_DIR))
DRIVE_DATA   = DRIVE_ROOT / "00_data"
DRIVE_MODELS = DRIVE_ROOT / "80_models"
LOCAL_DATA   = WORK_DIR / "00_data"
LOCAL_MODELS = WORK_DIR / "80_models"
LOCAL_EVAL   = WORK_DIR / "99_evaluation"
STAGE_DIR    = REPO_DIR / "50_modelling" / "02_2stage"


def sh(cmd, cwd=None, check=True):
    """Run a shell command, streaming output."""
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, cwd=cwd, check=check)


print("colab runtime:", IN_COLAB)

## 2. Mount Drive

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

assert DRIVE_ROOT.is_dir(), f"{DRIVE_ROOT} not found -- create it on your Drive first"
assert DRIVE_DATA.is_dir(), f"{DRIVE_DATA} not found -- upload 00_data there"
DRIVE_MODELS.mkdir(parents=True, exist_ok=True)
print("drive OK:", DRIVE_ROOT)

## 3. Clone the repo, wire up the directory layout

`main` is ~15 commits / 80 MB, so a plain clone is cheap.

In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
sh(f"git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")

for d in (LOCAL_DATA, LOCAL_MODELS, LOCAL_EVAL):
    d.mkdir(parents=True, exist_ok=True)

# 00_data / 80_models / 99_evaluation are gitignored, so recreate them at the repo
# root as symlinks into the local working dirs. Every relative path in the configs
# (../00_data/..., --out-dir 80_models/...) then resolves with no edits at all.
for name, target in (("00_data", LOCAL_DATA),
                     ("80_models", LOCAL_MODELS),
                     ("99_evaluation", LOCAL_EVAL)):
    link = REPO_DIR / name
    if link.is_symlink():
        link.unlink()
    elif link.exists():
        shutil.rmtree(link)
    link.symlink_to(target, target_is_directory=True)
    print(f"{link.name:14} -> {os.readlink(link)}")

## 4. Pull the data this notebook needs

In [ ]:
# Copy only what this notebook trains on, from Drive onto the local SSD. Training
# directly off the Drive mount is what you want to avoid: FUSE costs ~50-100 ms per
# file open, which starves the dataloader.
# Detection only needs the yolo_dataset_* trees (~450 MB), not the raw pages.
NEEDED = [
    "DIVA-HisDB/yolo_dataset_CB55",
    "DIVA-HisDB/yolo_dataset_CS18",
    "DIVA-HisDB/yolo_dataset_CS863",
    "U-DIADS-TL/yolo_dataset_latin14396",
    "U-DIADS-TL/yolo_dataset_latin2",
    "U-DIADS-TL/yolo_dataset_syr341",
]


def _files(root: Path):
    """relative path -> size, for every file under root (caches ignored)."""
    return {p.relative_to(root): p.stat().st_size
            for p in root.rglob("*") if p.is_file() and p.suffix != ".cache"}


def sync_data(force=False):
    """Copy Drive -> local, comparing file lists rather than directory existence.

    Deliberately not "copytree if missing": a copy interrupted by a dropped runtime
    leaves a directory that exists but is half-populated, and an existence check then
    skips it forever. Comparing actual files makes a re-run repair the gap.
    """
    total_new = total_mb = 0
    for rel in NEEDED:
        src, dst = DRIVE_DATA / rel, LOCAL_DATA / rel
        if not src.is_dir():
            print(f"[MISSING on Drive] {rel}")
            continue
        want = _files(src)
        have = _files(dst) if dst.is_dir() else {}
        todo = [r for r, sz in want.items() if force or have.get(r) != sz]
        if not todo:
            print(f"[ok]       {rel:44} {len(want):5} files")
            continue
        t0, mb = time.time(), 0.0
        for r in todo:
            d = dst / r
            d.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src / r, d)
            mb += want[r] / 1e6
        total_new += len(todo)
        total_mb += mb
        state = "copied  " if not have else "repaired"
        print(f"[{state}] {rel:44} {len(todo):5} files  {mb:8.1f} MB  in {time.time()-t0:5.1f}s")

    # Ultralytics label caches embed absolute paths from wherever they were built.
    for c in LOCAL_DATA.rglob("*.cache"):
        c.unlink()

    print(f"\ntransferred {total_new} files, {total_mb:.0f} MB")


sync_data()


## 5. Install dependencies

In [ ]:
# requirements.txt is shared with the local machine. Lines tagged `# [colab-skip]`
# (torch, numpy, tensorflow, ...) are dropped: Colab already ships them built against
# its CUDA, and reinstalling costs GBs and often breaks the driver match.
req = REPO_DIR / "requirements.txt"
if req.exists():
    kept = [ln for ln in req.read_text().splitlines()
            if ln.strip() and not ln.lstrip().startswith("#") and "[colab-skip]" not in ln]
    filtered = Path("/tmp/requirements-colab.txt")
    filtered.write_text("\n".join(kept) + "\n")
    print("installing:", ", ".join(k.split("==")[0] for k in kept))
    sh(f'pip -q install -r "{filtered}"')
else:
    raise SystemExit("requirements.txt missing from the clone -- commit and push it")

sh("nvidia-smi || true", check=False)

## 6. Checkpoint sync

`pull_models()` restores finished runs so they are skipped; `push_models()` sends results back and is also called after every individual run, so a dropped runtime costs you one run at most.

In [ ]:
# Only the artefacts worth keeping travel to Drive. Pushing whole run dirs (per-epoch
# weights, every visualisation) would exhaust a 15 GB Drive long before the matrix ends.
SYNC_GLOBS = ("weights/best.pt", "results.csv", "args.yaml")
MARKER = "weights/best.pt"          # its presence means "this run is finished"


def _copy_newer(src: Path, dst: Path) -> bool:
    if dst.exists() and dst.stat().st_size == src.stat().st_size \
       and dst.stat().st_mtime >= src.stat().st_mtime:
        return False
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    return True


def _sync(src_root: Path, dst_root: Path, label: str, only=None) -> int:
    if not src_root.is_dir():
        return 0
    n = 0
    roots = [src_root / only] if only else [src_root]
    for root in roots:
        if not root.is_dir():
            continue
        for pattern in SYNC_GLOBS:
            for src in root.rglob(pattern):
                n += _copy_newer(src, dst_root / src.relative_to(src_root))
    print(f"[{label}] {n} file(s)")
    return n


def pull_models():
    """Drive -> local, so finished runs are skipped instead of retrained."""
    return _sync(DRIVE_MODELS, LOCAL_MODELS, "pull")


def push_models(only=None):
    """Local -> Drive. Safe to call as often as you like."""
    return _sync(LOCAL_MODELS, DRIVE_MODELS, "push", only=only)


def is_done(out_rel: str, run: str) -> bool:
    return (LOCAL_MODELS / out_rel / run / MARKER).exists()


pull_models()

_stop = threading.Event()
if AUTOSYNC_MINUTES:
    def _loop():
        while not _stop.wait(AUTOSYNC_MINUTES * 60):
            try:
                push_models()
            except Exception as e:          # a Drive hiccup must not kill training
                print("autosync failed:", e)
    threading.Thread(target=_loop, daemon=True).start()
    print(f"autosync every {AUTOSYNC_MINUTES} min")

## 7. The training matrix

In [ ]:
# Mirrors 50_modelling/02_2stage/run_train_all.sh: the same 10 backbones, which
# deliberately track 01's list so the two matrices stay comparable. Transformer
# backbones route through hier_encoder's SFP neck, CNNs through stock YOLOv8 YAMLs.
BACKBONES = [
    "yolov8n",                             # stock CSPDarknet, COCO-pretrained
    "resnet34",                            # CNN, supervised ImageNet
    "resnet50",                            # CNN, supervised ImageNet
    "convnext",                            # CNN, supervised (convnext_tiny.in12k_ft_in1k)
    "convnext_dinov3",                     # CNN, DINOv3 SSL
    "vit_small_patch16_224.augreg_in21k",  # ViT-S/16 via SFP, supervised
    "vit_small_patch16_dinov3",            # ViT-S/16 via SFP, DINOv3 SSL
    "dinov2",                              # native DINOv2 ViT-S/14 via SFP
    "dinov2_reg",                          # native DINOv2 ViT-S/14 +registers via SFP
    "am-radio",                            # native AM-RADIO v2.5-B via SFP
]

FAMILIES = {
    "diva": dict(
        subsets=["CB55", "CS18", "CS863"],
        data={"CB55": "configs/data/diva_cb55_detect.yaml",
              "CS18": "configs/data/diva_cs18_detect.yaml",
              "CS863": "configs/data/diva_cs863_detect.yaml"},
        out="80_models/02_2stage/diva-hisdb/detection",
    ),
    "udiads": dict(
        subsets=["Latin14396", "Latin2", "Syr341"],
        data={"Latin14396": "configs/data/udiads_latin14396_detect.yaml",
              "Latin2": "configs/data/udiads_latin2_detect.yaml",
              "Syr341": "configs/data/udiads_syr341_detect.yaml"},
        out="80_models/02_2stage/u-diads-tl/detection",
    ),
}

EPOCHS, IMGSZ, BATCH, LR0, PATIENCE = 100, 1280, 2, 0.001, 50

PLAN = [(fam, ms, enc) for fam, spec in FAMILIES.items()
        for ms in spec["subsets"] for enc in BACKBONES]
print(f"{len(PLAN)} runs = {len(BACKBONES)} backbones x 6 subsets")

## 8. Sanity check — does everything resolve?

In [ ]:
os.chdir(STAGE_DIR)
sys.path.insert(0, str(STAGE_DIR))
import yaml

# Check the split directories the yaml actually names, not just the parent images/.
# A tree that exists but is missing images/train is exactly the failure that otherwise
# surfaces as an Ultralytics FileNotFoundError minutes into the first run.
ok = True
for fam, spec in FAMILIES.items():
    for ms, cfg in spec["data"].items():
        d = yaml.safe_load(open(STAGE_DIR / cfg))
        root = Path(d["path"])
        if not root.is_absolute():                    # same rule as resolve_data()
            root = (STAGE_DIR / cfg).parent.joinpath(root).resolve()
        for key in ("train", "val", "test"):
            if key not in d:
                continue
            split = root / d[key]
            n = sum(1 for p in split.iterdir()
                    if p.is_file() and p.suffix.lower() in (".jpg", ".jpeg", ".png", ".tif", ".tiff")
                    ) if split.is_dir() else 0
            lbl = root / d[key].replace("images", "labels", 1)
            nl = sum(1 for _ in lbl.glob("*.txt")) if lbl.is_dir() else 0
            good = n > 0 and nl > 0
            ok &= good
            print(f"{'OK  ' if good else 'FAIL'} {fam:7} {ms:12} {key:5} "
                  f"{n:4} images / {nl:4} labels  {split}")

assert ok, ("dataset splits are missing or empty -- re-run the data cell "
            "(sync_data(force=True) forces a full re-copy from Drive)")

# Stock YOLOv8 weights: train_detector.build_model() prefers this path and otherwise
# lets Ultralytics download them.
pre = DRIVE_ROOT / "80_models" / "02_2stage" / "pretrained" / "yolov8n.pt"
if pre.exists():
    dst = LOCAL_MODELS / "02_2stage" / "pretrained" / "yolov8n.pt"
    dst.parent.mkdir(parents=True, exist_ok=True)
    if not dst.exists():
        shutil.copy2(pre, dst)
    print("\npretrained:", dst)
else:
    print("\nno yolov8n.pt on Drive -- Ultralytics will download it")

import train_detector
print()
for i, (n, (kind, ref)) in enumerate(train_detector.BACKBONES.items(), 1):
    print(f"{i:2}. {n:38} [{kind}]")


## 9. Train

Runs sequentially, pushing each finished run to Drive before starting the next. A failed
run is reported and the loop continues.

The loop is written out here rather than shelling out to `run_train_all.sh` so that
`--project` can be passed as an absolute path — see the comment in the cell for why that
matters on Colab.

60 runs at 100 epochs will not finish in one Colab session. Just re-run the notebook:
completed runs come back from Drive and are skipped.

In [ ]:
os.chdir(STAGE_DIR)
done = skipped = failed = 0

for i, (fam, ms, enc) in enumerate(PLAN, 1):
    spec = FAMILIES[fam]
    run = f"yolov8_{enc}_{fam}_{ms}"
    out_rel = spec["out"].replace("80_models/", "")
    if is_done(out_rel, run):
        skipped += 1
        continue

    # Absolute --project on purpose. Ultralytics resolves a *relative* project against
    # its own RUNS_DIR setting, not the CWD (see cfg.get_save_dir), so the repo's
    # "../../80_models/..." only lands correctly on a machine whose RUNS_DIR happens to
    # sit two levels under the repo. An absolute path sidesteps that entirely.
    project = LOCAL_MODELS / out_rel

    print(f"\n{'='*80}\n[{i}/{len(PLAN)}] {run}  ({EPOCHS} epochs)\n{'='*80}")
    t0 = time.time()
    r = sh(
        f'{sys.executable} train_detector.py --encoder "{enc}" '
        f'--data "{spec["data"][ms]}" '
        f'--project "{project}" --name "{run}" '
        f'--epochs {EPOCHS} --imgsz {IMGSZ} --batch {BATCH} '
        f'--lr0 {LR0} --patience {PATIENCE}',
        check=False,
    )
    if r.returncode != 0:
        failed += 1
        print(f"[FAILED rc={r.returncode}] {run} -- continuing with the rest")
        continue

    done += 1
    print(f"[ok] {run} in {(time.time()-t0)/60:.1f} min")
    push_models(only=out_rel)

push_models()
print(f"\ntrained {done} | skipped {skipped} | failed {failed} | of {len(PLAN)}")

## 10. Summary

In [ ]:
import pandas as pd

rows = []
for fam, spec in FAMILIES.items():
    root = LOCAL_MODELS / spec["out"].replace("80_models/", "")
    for ms in spec["subsets"]:
        for enc in BACKBONES:
            csv = root / f"yolov8_{enc}_{fam}_{ms}" / "results.csv"
            if not csv.exists():
                continue
            d = pd.read_csv(csv)
            d.columns = [c.strip() for c in d.columns]
            col = next((c for c in d.columns if "mAP50(B)" in c), None)
            rows.append(dict(family=fam, subset=ms, encoder=enc, epochs=len(d),
                             mAP50=round(float(d[col].max()), 4) if col else None))

df = pd.DataFrame(rows)
print(df.to_string(index=False) if len(df) else "no results yet")
if len(df):
    print(f"\n{len(df)}/{len(PLAN)} runs complete")
    print(df.pivot_table(index="encoder", columns=["family", "subset"],
                         values="mAP50").to_string())